In [1]:
import os
import re
import zipfile
import io
import csv
from PIL import Image
from tqdm import tqdm

In [2]:
def extract_atlas_sprites(image_id: str, atlas_name: str):
    """
    Parses VT Lua atlas file and extracts individual sprites from the source
    :param image_id: The name of the image file (e.g., "EED797CC825CF42A").
    :param atlas_name: The name of the atlas type (e.g., "forge").
    """
    image_path = f"/content/{image_id}.png"
    atlas_path = f"/content/gui_{atlas_name}_atlas.lua"
    out_zip_path = f"/content/gui_{atlas_name}_atlas.zip"
    work_dir = f"/content/gui_{atlas_name}_atlas_temp"

    os.makedirs(work_dir, exist_ok=True)

    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    if not os.path.isfile(atlas_path):
        raise FileNotFoundError(f"Atlas Lua not found: {atlas_path}")

    img = Image.open(image_path).convert("RGBA")
    W, H = img.size
    print(f"Loaded image {image_path} size={W}x{H}")

    with open(atlas_path, "r", encoding="utf-8", errors="ignore") as f:
        lua_text = f.read()

    start_idx = lua_text.find(f"{atlas_name}_atlas")
    if start_idx == -1:
        raise ValueError(f"No header found for {atlas_name}_atlas (start_idx)")

    brace_open = lua_text.find("{", start_idx)
    if brace_open == -1:
        raise ValueError("Found header without {")

    depth = 0
    end_idx = None
    for i in range(brace_open, len(lua_text)):
        ch = lua_text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end_idx = i
                break

    if end_idx is None:
        raise ValueError("} absent")

    atlas_body = lua_text[brace_open+1:end_idx]

    entries = []
    pos = 0
    while pos < len(atlas_body):
        m = re.search(r'\S', atlas_body[pos:])
        if not m:
            break
        pos += m.start()

        # try to match identifier key
        key_match = re.match(r"""
            (?:
              (?P<ident>[A-Za-z0-9_\-]+)
            |
              \["(?P<qident>[^"]+)"\]
            )
            \s*=\s*{""", atlas_body[pos:], flags=re.X)

        if not key_match:
            pos += 1
            continue

        key = key_match.group("ident") or key_match.group("qident")

        # find { for entry
        entry_open = pos + key_match.end() - 1

        # find } for entry
        depth = 0
        entry_close = None
        for i in range(entry_open, len(atlas_body)):
            ch = atlas_body[i]
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    entry_close = i
                    break

        if entry_close is None:
            print(f"Warning: no closing for '{key}', skip")
            pos = entry_open + 1
            continue

        entry_text = atlas_body[entry_open+1:entry_close]
        entries.append((key, entry_text))
        pos = entry_close + 1

    print(f"Found {len(entries)} entries")

    size_re = re.compile(r'size\s*=\s*{\s*(\d+)\s*,\s*(\d+)\s*,?\s*}', flags=re.I)
    uv_re = re.compile(r'uv(00|11)\s*=\s*{\s*([0-9]*\.?[0-9]+)\s*,\s*([0-9]*\.?[0-9]+)\s*,?\s*}', flags=re.I)

    def clamp(v, a, b):
        return max(a, min(b, v))

    def sanitize_filename(name):
        return re.sub(r'[^A-Za-z0-9_\-\.]', '_', name)

    manifest_rows = []

    for key, body in tqdm(entries):
        # find size
        sm = size_re.search(body)
        uvs = {}
        for um in uv_re.finditer(body):
            idx = um.group(1)
            u = float(um.group(2))
            v = float(um.group(3))
            uvs[idx] = (u, v)

        if '00' not in uvs or '11' not in uvs:
            print(f"Skip '{key}': no uv00 or uv11")
            continue
        if not sm:
            print(f"Skip '{key}': no size")
            continue

        w_out = int(sm.group(1))
        h_out = int(sm.group(2))
        (u0, v0) = uvs['00']
        (u1, v1) = uvs['11']

        # convert to pixel coords
        x0 = int(round(u0 * W) - 1)
        y0 = int(round(v0 * H) - 1)
        x1 = int(round(u1 * W))
        y1 = int(round(v1 * H))

        x0 = clamp(x0, 0, W)
        x1 = clamp(x1, 0, W)
        y0 = clamp(y0, 0, H)
        y1 = clamp(y1, 0, H)

        # swap if needed
        if x1 < x0:
            x0, x1 = x1, x0
        if y1 < y0:
            y0, y1 = y1, y0

        if x1 == x0 or y1 == y0:
            print(f"Skip '{key}': ({x0},{y0})-({x1},{y1})")
            continue

        cropped = img.crop((x0, y0, x1, y1))

        safe = sanitize_filename(key)
        out_png = os.path.join(work_dir, f"{safe}.png")
        cropped.save(out_png, format="PNG")

        manifest_rows.append({
            "name": key,
            "file": f"{safe}.png",
            "pixel_rect": f"{x0},{y0},{x1},{y1}",
            "output_size": f"{w_out}x{h_out}"
        })

    print(f"\n{len(manifest_rows)} sprites saved to {work_dir}.")

    with zipfile.ZipFile(out_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for r in manifest_rows:
            full = os.path.join(work_dir, r["file"])
            zf.write(full, arcname=r["file"])

    print("Created zip:", out_zip_path)

In [3]:
extract_atlas_sprites("gui_frames_atlas", "frames")
extract_atlas_sprites("gui_hud_atlas", "hud")
extract_atlas_sprites("gui_loading_atlas", "loading")
extract_atlas_sprites("gui_menu_buttons_atlas", "menu_buttons")
extract_atlas_sprites("gui_popup_atlas", "popup")
extract_atlas_sprites("gui_settings_atlas", "settings")
extract_atlas_sprites("gui_startup_settings_atlas", "startup_settings")
extract_atlas_sprites("7A5A590C28ED1213", "store_menu")
extract_atlas_sprites("9BE664BBC010E46A", "pose_items")
extract_atlas_sprites("48DADB9B00ABD188", "carousel")
extract_atlas_sprites("96B29B5304554A6A", "splash")
extract_atlas_sprites("278E1B82B3B8790E", "icons")
extract_atlas_sprites("3115E8BF9BAFF15B", "menu_cinematics")
extract_atlas_sprites("44318894C23FDFFF", "belakor")
extract_atlas_sprites("BA5FEA2BF65D15BE", "items")
extract_atlas_sprites("BA69D3EF569190AC", "versus_rewards")
extract_atlas_sprites("CA2F81855E01E137", "level_images")
extract_atlas_sprites("D503781A9F2121BA", "achievement_icons")
extract_atlas_sprites("DE0A2CC628F6B50E", "menus")

Loaded image /content/gui_frames_atlas.png size=4096x4096
Found 201 entries


100%|██████████| 201/201 [00:05<00:00, 36.26it/s]  
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'menu_frame_09_divider.png'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'menu_frame_09_divider_vertical.png'
  return self._open_to_write(zinfo, force_zip64=force_zip64)



201 sprites saved to /content/gui_frames_atlas_temp.
Created zip: /content/gui_frames_atlas.zip
Loaded image /content/gui_hud_atlas.png size=4096x4096
Found 799 entries


100%|██████████| 799/799 [00:03<00:00, 259.94it/s]



799 sprites saved to /content/gui_hud_atlas_temp.
Created zip: /content/gui_hud_atlas.zip
Loaded image /content/gui_loading_atlas.png size=2048x1024
Found 87 entries


...


93 sprites saved to /content/gui_level_images_atlas_temp.
Created zip: /content/gui_level_images_atlas.zip
Loaded image /content/D503781A9F2121BA.png size=4096x4096
Found 880 entries


100%|██████████| 880/880 [00:06<00:00, 131.64it/s]



880 sprites saved to /content/gui_achievement_icons_atlas_temp.
Created zip: /content/gui_achievement_icons_atlas.zip
Loaded image /content/DE0A2CC628F6B50E.png size=4096x4096
Found 464 entries


100%|██████████| 464/464 [00:04<00:00, 94.63it/s]
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'icon_fire.png'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'icon_block.png'
  return self._open_to_write(zinfo, force_zip64=force_zip64)



464 sprites saved to /content/gui_menus_atlas_temp.
Created zip: /content/gui_menus_atlas.zip


In [4]:
import os
import zipfile
from google.colab import files

folder_to_search = '/content'
output_zip_name = 'all_zips_bundled.zip'

with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as master_zip:
    for filename in os.listdir(folder_to_search):
        file_path = os.path.join(folder_to_search, filename)
        if os.path.isfile(file_path) and filename.endswith('.zip') and filename != output_zip_name:
            master_zip.write(file_path, arcname=filename)
            print(f"Added to bundle: {filename}")

print(f"\nSuccessfully created {output_zip_name}")

files.download(output_zip_name)

Added to bundle: gui_menu_buttons_atlas.zip
Added to bundle: gui_icons_atlas.zip
Added to bundle: gui_hud_atlas.zip
Added to bundle: gui_menus_atlas.zip
Added to bundle: gui_store_menu_atlas.zip
Added to bundle: gui_frames_atlas.zip
Added to bundle: gui_items_atlas.zip
Added to bundle: gui_level_images_atlas.zip
Added to bundle: gui_pose_items_atlas.zip
Added to bundle: gui_achievement_icons_atlas.zip
Added to bundle: gui_startup_settings_atlas.zip
Added to bundle: gui_menu_cinematics_atlas.zip
Added to bundle: gui_loading_atlas.zip
Added to bundle: gui_versus_rewards_atlas.zip
Added to bundle: gui_splash_atlas.zip
Added to bundle: gui_belakor_atlas.zip
Added to bundle: gui_popup_atlas.zip
Added to bundle: gui_settings_atlas.zip
Added to bundle: gui_carousel_atlas.zip

Successfully created all_zips_bundled.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>